# 14 · pandas 3 Pitfalls

Old habits that break in pandas 3: chained assignment, inplace=True, and checking for object text columns.

In [1]:
# Setup: the steps from the previous parts (run this first)
import pandas as pd
pd.set_option("display.max_colwidth", 30)

URL = ("https://raw.githubusercontent.com/rfordatascience/tidytuesday/"
       "master/data/2021/2021-04-20/netflix_titles.csv")
df = pd.read_csv(URL)

df = df.drop(columns=["cast", "description"])

df["director"] = df["director"].fillna("Unknown")
df["country"] = df["country"].fillna("Unknown")
df = df.dropna(subset=["date_added", "rating"])

df["date_added"] = pd.to_datetime(df["date_added"].str.strip(), format="%B %d, %Y")
df["year_added"] = df["date_added"].dt.year
df["month_added"] = df["date_added"].dt.month_name()
df["duration_num"] = df["duration"].str.split(" ").str[0].astype(int)
df["main_country"] = df["country"].str.split(", ").str[0]

df["years_to_netflix"] = df["year_added"] - df["release_year"]
movies = df[df["type"] == "Movie"].assign(hours=pd.col("duration_num") / 60)

In [2]:
df[df["rating"] == "NR"]["rating"] = "Not Rated"

/var/folders/w1/tfjmbg0175n5h7drtfqqh7gw0000gn/T/ipykernel_83398/2945567065.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/docs/dev/user_guide/migration.html#chained-assignment
  df[df["rating"] == "NR"]["rating"] = "Not Rated"


In [3]:
(df["rating"] == "Not Rated").sum()

np.int64(0)

In [4]:
df.loc[df["rating"] == "NR", "rating"] = "Not Rated"
(df["rating"] == "Not Rated").sum()

np.int64(83)

In [5]:
df.dropna(inplace=True)

/var/folders/w1/tfjmbg0175n5h7drtfqqh7gw0000gn/T/ipykernel_83398/1379821321.py:1: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.dropna(inplace=True)


In [6]:
print(df["title"].dtype)
df["title"].dtype == object

str


False